In [ ]:
# 1. 필수 라이브러리 설치
!pip install transformers datasets scikit-learn pandas accelerate

import os
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback

# 2. 시드 고정 (결과 재현을 위해 필수)
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

# 3. 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Current Device: {device}")

Current Device: cuda


In [ ]:
# 4. 데이터 로드
# 파일 이름이 대소문자 정확한지 확인하세요 (Train.csv vs train.csv)
train_df = pd.read_csv("/content/Train.csv")
test_df = pd.read_csv("/content/Test.csv")
submission_df = pd.read_csv("/content/sample_submission.csv")

# Title과 Text를 합쳐서 하나의 입력으로 만듭니다
train_df['input_text'] = train_df['title'].fillna('') + " " + train_df['text'].fillna('')
test_df['input_text'] = test_df['title'].fillna('') + " " + test_df['text'].fillna('')

# 학습셋과 검증셋 분리 (8:2)
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)

print(f"학습 데이터 개수: {len(train_data)}")
print(f"검증 데이터 개수: {len(val_data)}")

학습 데이터 개수: 34718
검증 데이터 개수: 8680


In [ ]:
# 5. 토크나이저 및 데이터셋 정의
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class NewsDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        # 텍스트가 너무 짧은 경우를 대비해 예외 처리 (안전장치)
        if text is None or text == 'nan':
            text = ""

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        output = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

        if self.labels is not None:
            output['labels'] = torch.tensor(self.labels[item], dtype=torch.long)

        return output

# 데이터셋 생성 (여기서 max_len을 256으로 설정!)
train_dataset = NewsDataset(train_data['input_text'].to_numpy(), train_data['label'].to_numpy(), max_len=256)
val_dataset = NewsDataset(val_data['input_text'].to_numpy(), val_data['label'].to_numpy(), max_len=256)
test_dataset = NewsDataset(test_df['input_text'].to_numpy(), max_len=256)

# 6. 평가 함수 정의
def compute_metrics(p):
    pred, labels = p
    pred = np.argmax(pred, axis=1)
    f1 = f1_score(labels, pred, average='binary')
    acc = accuracy_score(labels, pred)
    return {'accuracy': acc, 'f1': f1}

# 7. 모델 초기화
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

# 8. 학습 파라미터 설정
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=32,  # <--- 길이를 줄였으니 16 -> 32로 늘림 (속도 UP)
    per_device_eval_batch_size=32,   # <--- 검증 속도도 UP
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    report_to="none",
    fp16=True  # <--- 고속 연산 모드 (속도 2배 향상)
)

# 9. Trainer 정의 및 학습 시작
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print(">>> 고속 모드로 학습 다시 시작!")
trainer.train()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


>>> 고속 모드로 학습 다시 시작!


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.223900,0.215485,0.847005,0.867306
2,0.219200,0.213348,0.845276,0.816955
3,0.221800,0.213371,0.847005,0.867306


TrainOutput(global_step=3255, training_loss=0.23234383631411784, metrics={'train_runtime': 1607.3622, 'train_samples_per_second': 64.798, 'train_steps_per_second': 2.025, 'total_flos': 1.370203442998272e+16, 'train_loss': 0.23234383631411784, 'epoch': 3.0})

In [ ]:
# 모델 저장
torch.save(model.state_dict(), 'best_model.pth')
print(">>> 모델 저장 완료 (best_model.pth)")

>>> 모델 저장 완료 (best_model.pth)


In [ ]:
# 10. 테스트 데이터 예측
print(">>> 테스트 데이터 예측 중...")
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)

# 11. 제출 파일 생성
submission_df['label'] = pred_labels
submission_df.to_csv('submission.csv', index=False)
print(">>> submission.csv 생성 완료! 파일을 다운로드하세요.")

>>> 테스트 데이터 예측 중...


>>> submission.csv 생성 완료! 파일을 다운로드하세요.


In [ ]:
from google.colab import files

# 모델 파일 다운로드 (스마트리드 제출용)
files.download('best_model.pth')

# 결과 파일 다운로드 (캐글 제출용)
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>